# §11.1.5 — 동일 파라미터 예산에서의 표본 효율 비교

> 딥러닝 교재 · 3부 11장 1절 5항 (🐍)
> 선행: §11.1.2(평행이동 등변) · §11.1.3(토플리츠 구조와 파라미터 수) · §1.4.4(구조와 표본 효율)
> 부록 K(노트북 사용 안내)

## 이 노트북이 답하는 질문

1. **파라미터 수를 같게 맞추면** 합성곱망과 완전연결망의 격차는 얼마나 남는가? §1.4.4는 무작위 특징이었지만 여기서는 둘 다 끝까지 학습한다.
2. **완전연결망에 파라미터를 30배 더 주면** 격차가 닫히는가 — 병목이 용량인가 편향인가?
3. **화소 순열 대조**: 같은 정보량의 데이터에서 합성곱의 이점만 사라지는가?

**예상 실행 시간** CPU 약 3분 (`FAST = True`이면 약 1분).
이 실험을 이해하려면 §11.1.3의 파라미터 수 계산과 §1.4.2의 귀납 편향 삼분할이 필요합니다. 코드가 유도를 대체하지 않습니다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — §1.4.4의 막대 무늬 과제를 그대로

$10\times10$ 표준정규 잡음 위에, **무작위 위치**에 $3\times3$ 무늬(가로 막대 / 세로 막대)를 **무작위 부호** $s\in\{-1,+1\}$로 찍는다.
국소성 · 위치 무작위성 · 부호 무작위성이 각각 지역성 · 가중치 공유 · 비선형성과 짝을 이룬다 (§1.4.4).

In [ ]:
IMG = 10
AMP = 4.0            # 무늬 진폭
T_H = np.zeros((3, 3)); T_H[1, :] = 1.0          # 가로 막대
T_V = np.zeros((3, 3)); T_V[:, 1] = 1.0          # 세로 막대

def make_data(n, rn):
    X = rn.standard_normal((n, IMG, IMG))
    y = rn.integers(0, 2, n)                      # 0: 가로, 1: 세로
    s = rn.choice([-1.0, 1.0], n)
    r = rn.integers(0, IMG - 2, n); c = rn.integers(0, IMG - 2, n)
    for i in range(n):
        T = T_V if y[i] else T_H
        X[i, r[i]:r[i]+3, c[i]:c[i]+3] += AMP * s[i] * T
    return X, y.astype(float)

Xte, yte = make_data(4000, np.random.default_rng(SEED + 999))
print("시험 표본:", Xte.shape)

---
## 2. 세 모델 — 원리 그대로의 NumPy 구현

| 모델 | 구조 | 어디를 제약했는가 |
|---|---|---|
| CNN | $3\times3$ 합성곱 $F$개 → ReLU → 전역 max·avg 풀링 → 선형 | 지역성 + 가중치 공유 (§11.1.3) |
| MLP | $100 \to H \to 1$ | 제약 없음 (완전연결) |
| 선형 | $100 \to 1$ | 비선형 없음 — §1.4.4의 명제에 의해 우연 수준 |

학습은 셋 모두 **같은 손실(로지스틱) · 같은 최적화기(Adam) · 같은 걸음 수**다. 다른 것은 가설 공간뿐이다.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def im2col(X, k=3):
    # X: (N,H,W) -> (N, H-k+1, W-k+1, k*k)
    v = sliding_window_view(X, (k, k), axis=(1, 2))
    return v.reshape(X.shape[0], X.shape[1]-k+1, X.shape[2]-k+1, k*k)

def adam_step(p, g, m, v, t, lr=3e-3, b1=0.9, b2=0.999, eps=1e-8):
    m[:] = b1*m + (1-b1)*g
    v[:] = b2*v + (1-b2)*g*g
    mh = m/(1-b1**t); vh = v/(1-b2**t)
    p -= lr*mh/(np.sqrt(vh)+eps)

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

class CNN:
    # conv(1->F,3x3) -> ReLU -> [global max ; global avg] -> linear(2F->1)
    def __init__(self, F, rn):
        self.F = F
        self.W = rn.standard_normal((9, F)) * np.sqrt(2/9)
        self.b = np.zeros(F)
        self.u = rn.standard_normal(2*F) * np.sqrt(1/(2*F))
        self.c = np.zeros(1)
        self.params = [self.W, self.b, self.u, self.c]
    def n_params(self):
        return sum(p.size for p in self.params)
    def forward(self, X):
        col = im2col(X)                          # (N,8,8,9)
        Z = col @ self.W + self.b                # (N,8,8,F)
        A = np.maximum(Z, 0)
        Amax = A.max(axis=(1, 2))                # (N,F)
        Aavg = A.mean(axis=(1, 2))
        feat = np.concatenate([Amax, Aavg], axis=1)
        out = feat @ self.u + self.c
        self.cache = (col, Z, A, Amax, feat, X.shape)
        return out
    def backward(self, dout):
        col, Z, A, Amax, feat, xs = self.cache
        N = xs[0]; F = self.F
        du = feat.T @ dout; dc = np.array([dout.sum()])
        dfeat = np.outer(dout, self.u)           # (N,2F)
        dAmax, dAavg = dfeat[:, :F], dfeat[:, F:]
        dA = np.zeros_like(A)
        # avg 경로
        dA += dAavg[:, None, None, :] / (A.shape[1]*A.shape[2])
        # max 경로 — argmax 위치로만
        flat = A.reshape(N, -1, F)
        idx = flat.argmax(axis=1)                # (N,F)
        dflat = np.zeros_like(flat)
        n_i = np.arange(N)[:, None]; f_i = np.arange(F)[None, :]
        dflat[n_i, idx, f_i] += dAmax
        dA += dflat.reshape(A.shape)
        dZ = dA * (Z > 0)
        dW = col.reshape(-1, 9).T @ dZ.reshape(-1, F)
        db = dZ.sum(axis=(0, 1, 2))
        return [dW, db, du, dc]

class MLP:
    def __init__(self, H, rn):
        self.W1 = rn.standard_normal((IMG*IMG, H)) * np.sqrt(2/(IMG*IMG))
        self.b1 = np.zeros(H)
        self.W2 = rn.standard_normal(H) * np.sqrt(1/max(H, 1))
        self.b2 = np.zeros(1)
        self.params = [self.W1, self.b1, self.W2, self.b2]
    def n_params(self):
        return sum(p.size for p in self.params)
    def forward(self, X):
        Xf = X.reshape(X.shape[0], -1)
        Z1 = Xf @ self.W1 + self.b1
        A1 = np.maximum(Z1, 0)
        out = A1 @ self.W2 + self.b2
        self.cache = (Xf, Z1, A1)
        return out
    def backward(self, dout):
        Xf, Z1, A1 = self.cache
        dW2 = A1.T @ dout; db2 = np.array([dout.sum()])
        dA1 = np.outer(dout, self.W2)
        dZ1 = dA1 * (Z1 > 0)
        dW1 = Xf.T @ dZ1; db1 = dZ1.sum(axis=0)
        return [dW1, db1, dW2, db2]

class Linear:
    def __init__(self, rn):
        self.W = rn.standard_normal(IMG*IMG) * 0.01
        self.b = np.zeros(1)
        self.params = [self.W, self.b]
    def n_params(self):
        return sum(p.size for p in self.params)
    def forward(self, X):
        Xf = X.reshape(X.shape[0], -1)
        self.cache = Xf
        return Xf @ self.W + self.b
    def backward(self, dout):
        Xf = self.cache
        return [Xf.T @ dout, np.array([dout.sum()])]

def train_eval(model, Xtr, ytr, Xev, yev, steps=350, batch=128, seed=0):
    ms = [np.zeros_like(p) for p in model.params]
    vs = [np.zeros_like(p) for p in model.params]
    rb = np.random.default_rng(seed)
    n = len(ytr)
    for t in range(1, steps+1):
        idx = rb.integers(0, n, min(batch, n))
        z = model.forward(Xtr[idx])
        p = sigmoid(z.ravel())
        dz = (p - ytr[idx]) / len(idx)           # 로지스틱 손실의 기울기 (§2.3.2)
        gs = model.backward(dz)
        for pp, g, m, v in zip(model.params, gs, ms, vs):
            adam_step(pp, g, m, v, t)
    hits = 0
    for s0 in range(0, len(yev), 1000):          # 조각내어 평가 (메모리 절약)
        sl = slice(s0, s0+1000)
        hits += np.sum((model.forward(Xev[sl]).ravel() > 0) == (yev[sl] > 0.5))
    return hits / len(yev)

# 파라미터 예산 맞추기
F_CNN = 48                       # CNN: 9F+F+2F+1
H_EQ  = 5                        # MLP(동일 예산): 101H+H+... ≈ CNN
H_BIG = 180                      # MLP(30배 예산)
_r = np.random.default_rng(0)
print("파라미터 수  CNN:", CNN(F_CNN, _r).n_params(),
      "| MLP(동일):", MLP(H_EQ, _r).n_params(),
      "| MLP(대형):", MLP(H_BIG, _r).n_params(),
      "| 선형:", Linear(_r).n_params())

---
## 3. 표본 효율 — $n$을 훑는다

In [ ]:
NS    = [25, 100, 400, 1600] if FAST else [25, 50, 100, 200, 400, 800, 1600]
SEEDS = 2 if FAST else 3
MODELS = {'CNN': lambda rn: CNN(F_CNN, rn),
          'MLP-동일': lambda rn: MLP(H_EQ, rn),
          'MLP-대형': lambda rn: MLP(H_BIG, rn),
          '선형': lambda rn: Linear(rn)}

acc = {k: np.zeros((len(NS), SEEDS)) for k in MODELS}
for si in range(SEEDS):
    rn = np.random.default_rng(SEED + si)
    Xpool, ypool = make_data(max(NS), rn)
    for ni, n in enumerate(NS):
        for name, ctor in MODELS.items():
            model = ctor(np.random.default_rng(SEED + 7*si + (ni+3)*13))
            acc[name][ni, si] = train_eval(model, Xpool[:n], ypool[:n], Xte, yte, seed=si*100+ni)
    print(f"seed {si+1}/{SEEDS} 완료  ({time.time()-_t0:.0f}초)")

for name in MODELS:
    print(name, np.round(acc[name].mean(axis=1), 3))

> **읽는 법.** CNN은 수십 개의 표본으로 끝난다. 동일 예산 MLP는 수천 개로도 못 미치고,
> 파라미터를 30배 준 MLP도 CNN의 소표본 성능을 따라잡지 못한다.
> **격차의 병목은 용량이 아니라 편향이다.**

---
## 4. 화소 순열 대조 — 정보는 그대로, 이웃 구조만 파괴

모든 이미지에 **같은** 순열을 적용한다. 순열은 가역이므로 베이즈 위험은 변하지 않는다 (§1.2.6).
합성곱의 가정(이웃이 의미 있다)만 정확히 저격한다.

In [ ]:
perm = np.random.default_rng(SEED + 123).permutation(IMG*IMG)
def permute(X):
    return X.reshape(len(X), -1)[:, perm].reshape(X.shape)

NS_P = [100, 400, 1600] if FAST else [50, 100, 400, 1600]
accP = {k: np.zeros((len(NS_P), SEEDS)) for k in ['CNN', 'MLP-동일']}
XteP = permute(Xte)
for si in range(SEEDS):
    rn = np.random.default_rng(SEED + si)
    Xpool, ypool = make_data(max(NS_P), rn)
    XpoolP = permute(Xpool)
    for ni, n in enumerate(NS_P):
        accP['CNN'][ni, si] = train_eval(CNN(F_CNN, np.random.default_rng(3*si)), XpoolP[:n], ypool[:n], XteP, yte, seed=si)
        accP['MLP-동일'][ni, si] = train_eval(MLP(H_EQ, np.random.default_rng(5*si)), XpoolP[:n], ypool[:n], XteP, yte, seed=si)
print("순열 후 CNN :", np.round(accP['CNN'].mean(axis=1), 3))
print("순열 후 MLP :", np.round(accP['MLP-동일'].mean(axis=1), 3))

---
## 5. 교재 그림 — fig_11_1_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 표본 예시
ax = axes[0]
Xs, ys = make_data(4, np.random.default_rng(4))
tile = np.concatenate([np.concatenate([Xs[0], Xs[1]], axis=1),
                       np.concatenate([Xs[2], Xs[3]], axis=1)], axis=0)
im = ax.imshow(tile, cmap='gray'); ax.grid(False)
ax.axhline(IMG-0.5, color=CB[4], lw=0.8); ax.axvline(IMG-0.5, color=CB[4], lw=0.8)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(lab('(a) 과제 표본 네 개 (가로/세로 막대)', '(a) task samples'), fontsize=10)

# (b) 표본 효율
ax = axes[1]
styles = {'CNN': (CB[5], '-o'), 'MLP-동일': (CB[1], '-s'),
          'MLP-대형': (CB[3], '-^'), '선형': (CB[0], '--x')}
labs_en = {'CNN': 'CNN', 'MLP-동일': 'MLP (equal)', 'MLP-대형': 'MLP (30x)', '선형': 'linear'}
for name in MODELS:
    c, st = styles[name]
    m = acc[name].mean(axis=1); sd = acc[name].std(axis=1)
    ax.plot(NS, m, st, color=c, label=lab(name, labs_en[name]), ms=4)
    ax.fill_between(NS, m-sd, m+sd, color=c, alpha=0.15)
ax.set_xscale('log'); ax.axhline(0.5, color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('학습 표본 수 $n$ (개)', 'training samples $n$'))
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('(b) 동일 예산 표본 효율', '(b) sample efficiency at equal budget'), fontsize=10)
ax.legend(fontsize=8, loc='center left')

# (c) 목표 도달 표본 수 (로그 보간, 미도달은 상한 표시)
ax = axes[2]
TARGET = 0.8
def n_to_reach(m, ns, target):
    for i in range(len(ns)):
        if m[i] >= target:
            if i == 0:
                return float(ns[0])
            # 로그-n 선형 보간
            f = (target - m[i-1]) / (m[i] - m[i-1])
            return float(np.exp(np.log(ns[i-1]) + f*(np.log(ns[i]) - np.log(ns[i-1]))))
    return np.nan
names3 = ['CNN', 'MLP-대형', 'MLP-동일']
vals = [n_to_reach(acc[k].mean(axis=1), NS, TARGET) for k in names3]
ymax = max(NS)*2.5
for i, (k, v) in enumerate(zip(names3, vals)):
    if np.isfinite(v):
        ax.bar(i, v, color=styles[k][0], alpha=0.85)
        ax.text(i, v*1.15, f'{v:.0f}', ha='center', fontsize=9)
    else:
        ax.bar(i, ymax, color=styles[k][0], alpha=0.3, hatch='//')
        ax.text(i, ymax*0.55, lab(f'>{max(NS)}\n미도달', f'>{max(NS)}'), ha='center', fontsize=8)
ax.set_xticks(range(len(names3)))
ax.set_xticklabels([lab(k, labs_en[k]) for k in names3], fontsize=9)
ax.set_yscale('log'); ax.set_ylim(10, ymax*1.3)
ax.set_ylabel(lab(f'정확도 {TARGET:.0%} 도달에 필요한 $n$', f'$n$ to reach {TARGET:.0%}'))
ax.set_title(lab('(c) 목표 도달 비용', '(c) cost to reach target'), fontsize=10)

# (d) 순열 대조
ax = axes[3]
for name in ['CNN', 'MLP-동일']:
    c, st = styles[name]
    ax.plot(NS, acc[name].mean(axis=1), st, color=c, alpha=0.3, ms=3,
            label=lab(name+' (원본)', labs_en[name]+' (orig)'))
    ax.plot(NS_P, accP[name].mean(axis=1), st, color=c, ms=4,
            label=lab(name+' (순열)', labs_en[name]+' (perm)'))
ax.set_xscale('log'); ax.axhline(0.5, color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('학습 표본 수 $n$ (개)', 'training samples $n$'))
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('(d) 화소 순열 대조 — CNN의 이점만 사라진다', '(d) pixel permutation control'), fontsize=10)
ax.legend(fontsize=7)

save_book_fig(fig, 'fig_11_1_5')
plt.show()

> ### 이 그림이 §11.1.5의 결론이다
>
> (b) 동일 파라미터에서 곡선 사이 간격은 **데이터로 메울 수 없는 종류**의 간격이다.
> (c) 같은 목표에 도달하는 비용이 표본 수십 개 vs 수천 개로 갈린다 — 가중치 공유가 산 절약의 실측값.
> (d) 순열은 베이즈 위험을 바꾸지 않지만 CNN의 이점만 지운다.
> **구조의 이점은 모델의 성질이 아니라 모델과 문제의 관계다.**

---
## 6. 자기 점검

1. (b)에서 MLP-대형이 MLP-동일보다 나은 구간은 어디인가? 그 구간에서도 CNN에 못 미치는 이유를 §1.4.2의 언어로 설명하라.
2. `AMP`를 2.0으로 낮추면 세 곡선의 간격은 어떻게 되겠는가? 예측한 뒤 실행해 확인하라.
3. (d)에서 순열 후에도 CNN이 우연보다 나은 이유는? (힌트: 순열이 모든 국소 통계를 파괴하는가?)
4. 이 실험의 CNN은 풀링으로 위치 정보를 버렸다 (§11.3.5). 과제를 "무늬가 왼쪽 절반에 있는가"로 바꾸면 어떤 구조가 필요한가?

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `AMP` | 1절 | 4.0 | 무늬 대비. 낮추면 모든 곡선이 오른쪽으로 밀린다 |
| `F_CNN`, `H_EQ`, `H_BIG` | 2절 | 48/5/180 | 파라미터 예산. 예산 정합성을 출력으로 확인할 것 |
| `NS`, `SEEDS` | 3절 | — | 표본 수 격자와 반복 수 |
| `TARGET` | 5절 | 0.85 | (c)의 목표 정확도 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")